# 성격이 다른 세 에이전트가 영업 메일 쓰기

수업(2026-10-15 오후) 내용:

- 같은 제품(SNIPER APT) 소개 메일을 **세 명의 에이전트**가 씁니다.
  1. **정직한** 에이전트: 과장하지 않고, 못 하는 건 못 한다고 말한다.
  2. **유머러스한** 에이전트: 가볍고 재미있고 읽는 데 부담이 없다.
  3. **공격적인** 에이전트: "도입하지 않으면 회사가 망할 수 있다"는 식으로 쓴다.
- 가장 좋은 메일을 골라 **Gmail SMTP로 자기 자신에게** 보냅니다.

준비물(`.env`):

```
EMAIL_SMTP_SERVER=smtp.gmail.com
EMAIL_APP_PASSWORD=구글 앱 비밀번호 16자리
EMAIL_ADDRESS=자기 이메일
```

In [1]:
import asyncio
import os
import smtplib
from email.mime.text import MIMEText

from dotenv import load_dotenv
from agents import Agent, Runner, function_tool

load_dotenv(r"c:\aiwins\.env", override=True)  # Downloads에는 .env가 없으므로 경로를 직접 준다

MODEL = "gpt-4.1-mini"
PRODUCT = "윈스 SNIPER APT (지능형 지속 위협 탐지·대응 보안 솔루션)"

## 1. 성격이 다른 세 에이전트

In [2]:
common = f"당신은 {PRODUCT}의 영업 담당자입니다. 고객사 보안 담당자에게 보낼 콜드 메일을 한국어로 씁니다. 제목과 본문을 쓰세요."

honest_agent = Agent(
    name="정직한 영업",
    instructions=common + " 과장하지 않고 사실만 씁니다. 제품이 못 하는 것은 못 한다고 솔직하게 밝힙니다.",
    model=MODEL,
)
humorous_agent = Agent(
    name="유머러스한 영업",
    instructions=common + " 가볍고 재미있게, 읽는 데 부담이 없도록 씁니다.",
    model=MODEL,
)
aggressive_agent = Agent(
    name="공격적인 영업",
    instructions=common + " 매우 공격적으로 씁니다. 지금 도입하지 않으면 해킹 사고로 회사가 망할 수 있다고 강하게 경고합니다.",
    model=MODEL,
)

writers = [honest_agent, humorous_agent, aggressive_agent]

## 2. 세 명에게 동시에 쓰게 하기

`asyncio.gather`로 세 에이전트를 **동시에** 실행합니다. 하나씩 기다리는 것보다 빠릅니다.

In [3]:
request = "보안 담당자에게 보낼 콜드 메일을 써 주세요."
results = await asyncio.gather(*[Runner.run(agent, request) for agent in writers])
drafts = [r.final_output for r in results]

for agent, draft in zip(writers, drafts):
    print(f"========== {agent.name} ==========")
    print(draft)
    print()

========== 정직한 영업 ==========
제목: 지능형 위협 탐지·대응 솔루션 윈스 SNIPER APT 소개드립니다

안녕하세요, [고객사명] 보안 담당자님.

저는 윈스의 SNIPER APT 영업 담당자 [이름]입니다.

최근 지능형 지속 위협(APT) 공격이 고도화됨에 따라, 이를 탐지하고 효과적으로 대응하는 보안 솔루션의 필요성이 확대되고 있습니다. 윈스 SNIPER APT는 네트워크 트래픽 분석을 통해 이상행위를 탐지하며, 공격의 전 단계에 걸친 위협 정보를 수집·분석하여 대응을 지원하는 제품입니다.

다만, SNIPER APT는 엔드포인트 내부의 상세 행위 분석이나 악성 코드 제거 기능은 포함되어 있지 않습니다. 따라서 기존 엔드포인트 보안 솔루션과의 연동을 통해 보완적으로 활용하실 때 효과적입니다.

관심 있으시면 제품 데모나 상담 일정을 조율해 드리겠습니다. 편하신 시간 알려주시면 감사하겠습니다.

감사합니다.

윈스 SNIPER APT 담당  
[이름]  
[연락처]  
[이메일]

========== 유머러스한 영업 ==========
제목: “해커야, 거기서 잠깐! 우리 보안 좀 믿어주세요😉”

안녕하세요, [고객사명] 보안 담당자님!

요즘 해커들이 참 바쁜가 봐요. 매일 새로운 공격 시도에 마음 졸이시죠? 저희 윈스 SNIPER APT가 그 고민, 살짝 덜어드릴까 합니다.

SNIPER APT는 ‘스텔스 모드 해커’도 찾아내는 지능형 지속 위협 탐지·대응 솔루션이에요. 말 그대로 언제 어디서나 우리 네트워크를 지켜주는 ‘보안 스나이퍼’죠!

가볍게 한 번 만나서, 커피 한 잔과 함께 어떻게 보안을 한 단계 업그레이드할 수 있을지 이야기 나눠볼까요? 부담 없이 연락 주세요!

좋은 하루 되세요!  
[내 이름] 드림  
윈스 SNIPER APT 영업팀  
[연락처]  
[이메일]

========== 공격적인 영업 ==========
제목: 지금 즉시 윈스 SNIPER APT 도입하지 않으면 회사는 해킹으로 파산합니다!

안녕하세

## 3. 가장 좋은 메일 고르기 (심사 에이전트)

In [4]:
picker = Agent(
    name="메일 심사",
    instructions="여러 영업 메일 중 고객이 가장 답장할 것 같은 메일 하나를 고릅니다. "
                 "고른 메일의 번호와 이유를 한두 문장으로 쓰고, 그 메일의 제목과 본문을 그대로 붙이세요.",
    model=MODEL,
)

numbered = "\n\n".join(f"[{i + 1}번 - {a.name}]\n{d}" for i, (a, d) in enumerate(zip(writers, drafts)))
best = await Runner.run(picker, numbered)
print(best.final_output)

2번 메일이 답장할 가능성이 가장 높습니다. 유머러스하고 친근한 어투로 부담을 줄여주며, 만나서 이야기 나누자는 제안이 자연스럽고 긍정적 반응을 이끌어내기 좋습니다.

제목: “해커야, 거기서 잠깐! 우리 보안 좀 믿어주세요😉”

안녕하세요, [고객사명] 보안 담당자님!

요즘 해커들이 참 바쁜가 봐요. 매일 새로운 공격 시도에 마음 졸이시죠? 저희 윈스 SNIPER APT가 그 고민, 살짝 덜어드릴까 합니다.

SNIPER APT는 ‘스텔스 모드 해커’도 찾아내는 지능형 지속 위협 탐지·대응 솔루션이에요. 말 그대로 언제 어디서나 우리 네트워크를 지켜주는 ‘보안 스나이퍼’죠!

가볍게 한 번 만나서, 커피 한 잔과 함께 어떻게 보안을 한 단계 업그레이드할 수 있을지 이야기 나눠볼까요? 부담 없이 연락 주세요!

좋은 하루 되세요!  
[내 이름] 드림  
윈스 SNIPER APT 영업팀  
[연락처]  
[이메일]


## 4. Gmail로 보내기

`send_email`을 **도구**로 만들어 에이전트가 직접 보내게 합니다. 받는 사람은 자기 자신(`EMAIL_ADDRESS`)입니다.

`.env`에 `EMAIL_APP_PASSWORD`가 비어 있으면 보내지 않고 알려 줍니다.

In [5]:
@function_tool
def send_email(subject: str, body: str) -> str:
    """메일을 자기 자신에게 보낸다.

    Args:
        subject: 메일 제목
        body: 메일 본문
    """
    server = os.getenv("EMAIL_SMTP_SERVER", "smtp.gmail.com")
    address = os.getenv("EMAIL_ADDRESS")
    password = os.getenv("EMAIL_APP_PASSWORD")
    if not (address and password):
        return "EMAIL_ADDRESS 또는 EMAIL_APP_PASSWORD가 .env에 없어 보내지 못했습니다."

    msg = MIMEText(body, "plain", "utf-8")
    msg["Subject"] = subject
    msg["From"] = address
    msg["To"] = address
    with smtplib.SMTP_SSL(server, 465) as smtp:
        smtp.login(address, password)
        smtp.send_message(msg)
    return f"{address}로 보냈습니다."


sender = Agent(
    name="메일 발송",
    instructions="받은 메일의 제목과 본문을 나눠 send_email 도구로 보내고, 결과를 한 줄로 알려 주세요.",
    tools=[send_email],
    model=MODEL,
)

result = await Runner.run(sender, best.final_output)
print(result.final_output)

메일 발송에 필요한 정보가 없어 메일을 보내지 못했습니다. 메일 발송에 필요한 설정(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)을 제공해 주시면 다시 시도하겠습니다.
